In [ ]:
# Configuration: the tables to copy from bronze to silver.# To load only a subset, edit this list.table_names = ['product', 'purchase_order_detail', 'purchase_order_header', 'sales_order_detail', 'sales_territory', 'vendor']source_schema = 'adventure_works.bronze'target_schema = 'adventure_works.silver'

In [ ]:
from pyspark.sql import functions as F# Read each bronze table, trim whitespace on string columns and overwrite the silver table.for name in table_names: df = spark.table(f'{source_schema}.{name}'); df = df.select([F.trim(F.col(c)).alias(c) if t == 'string' else F.col(c) for c, t in df.dtypes]); df.write.mode('overwrite').option('overwriteSchema', True).saveAsTable(f'{target_schema}.{name}'); print(name, 'rows written:', df.count())

In [ ]:
# Verification: bronze and silver row counts must match for every table.results = [(name, spark.table(f'{source_schema}.{name}').count(), spark.table(f'{target_schema}.{name}').count()) for name in table_names]verification = spark.createDataFrame(results, ['table', 'bronze_rows', 'silver_rows'])verification = verification.withColumn('counts_match', F.col('bronze_rows') == F.col('silver_rows'))display(verification)assert verification.filter(F.col('counts_match') == False).count() == 0, 'Bronze and silver counts disagree - check the load cell.'print('Success: all bronze and silver row counts match.')